# This file serves to postprocess model data for piControl

author: Eike E. Köhn....... 
date: Aug 20, 2025....... 
instructions: can be run with environment "conda env:pangeo-meso-2024.01.22" on spirit1

## Import packages

In [1]:
print('Loading packages...')
import sys
sys.path.append('../00_modules/.')
from import_packages import PackageGetter
globals().update(PackageGetter.import_standard_packages_for_analysis_and_plotting())
globals().update(PackageGetter.import_custom_packages())
misc_params = Params.additional_misc_params()

from datetime import date
from dateutil.relativedelta import relativedelta

from netCDF4 import Dataset


Loading packages...


## Set the parameters for the analysis of the 1pctCO2-cdr run

In [2]:
print('Setting parameters...')
run_params = Params.standard_set_piControl() #Params.standard_set_1pctCO2cdr()  #Params.test_set_1pctCO2cdr() #

Setting parameters...


## Now postprocess the model output, by looping over the models

### 1. set the variable to be treated

In [3]:
variable = 'talk'

if variable in ['hur','thetao','so','dissic','talk','po4']:
    depth_to_analyze = '1' # None
else:
    depth_to_analyze = ''

### 2. get the raw data paths (for 1pctco2, 1pctco2-cdr, and if available piControl)

In [4]:
def get_domain(variable):
    if variable in ['psl','hurs','hur']:
        domain = 'A'
    elif variable in ['siconc']:
        domain = 'SI'
    else:
        domain = 'O'
    return domain

def get_temporal_resolution(rparam,experiment,variable):
    if variable in ['dissic'] and experiment in ['1pctCO2','1pctCO2-cdr'] and rparam.model in ['CESM2','ACCESS-ESM1-5']:
        temporal_resolution = 'yr'
    else:
        temporal_resolution = 'mon'
    return temporal_resolution

def get_unit(variable):
    if variable in ['tos','thetao']:
        unit = '°C'
    elif variable in ['sos','so']:
        unit = 'psu'
    elif variable == 'psl':
        unit = 'Pa'
    elif variable in ['hurs','hur']:
        unit = '%'
    elif variable in ['dissic','talk']:
        unit = 'mol m-3'
    elif variable in ['fgco2']:
        unit = 'kg m-2 s-1'
    return unit

def get_raw_filepaths(run_params,variable):
    root_dir = '/data/ekoehn/CMIP6_derived'
    raw_filepaths = dict()
    for key in run_params.keys():

        rparam = run_params[key]

        # get the domain
        domain = get_domain(variable)

        # put the paths into the dictionary
        raw_filepaths_key = dict()

        # get the data paths
        experiments = ['piControl','1pctCO2']

        for experiment in experiments:
            # get the temporal resolution
            temporal_resolution = get_temporal_resolution(rparam,experiment,variable)
            filepath = f'{root_dir}/{variable}{depth_to_analyze}/{experiment}/{domain}{temporal_resolution}'
            filename = f'{variable}{depth_to_analyze}_{domain}{temporal_resolution}_{rparam.model}_{experiment}_{rparam.member}_gr_*.nc'
            #print(f'{filepath}/{filename}')
            print(f'{filepath}/{filename}')
            available_paths = glob.glob(f'{filepath}/{filename}')
            if len(available_paths)==0:
                print(f'{experiment}, {rparam.model}: No file available')
                raw_filepaths_key[experiment] = None
            elif len(available_paths)>1:
                print(f'{experiment}, {rparam.model}: More than one file available')
                raw_filepaths_key[experiment] = None
            else:
                raw_filepaths_key[experiment] = available_paths[0]
        
        raw_filepaths[key] = raw_filepaths_key
    return raw_filepaths

raw_data_paths = get_raw_filepaths(run_params,variable)

/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_UKESM1-0-LL_piControl_r1i1p1f2_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_UKESM1-0-LL_1pctCO2_r1i1p1f2_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_NorESM2-LM_piControl_r1i1p1f1_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_NorESM2-LM_1pctCO2_r1i1p1f1_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_MIROC-ES2L_piControl_r1i1p1f2_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_MIROC-ES2L_1pctCO2_r1i1p1f2_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_CNRM-ESM2-1_piControl_r1i1p1f2_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_CNRM-ESM2-1_1pctCO2_r1i1p1f2_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_ACCESS-ESM1-5_piControl_r1i1p1f1_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_ACCESS-ESM1-5_1pctCO2_r1i1p1f1_gr_*.nc
/data/ekoehn/CMIP6_derived/talk1/piContr

In [5]:
raw_data_paths

{'2': {'piControl': '/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_UKESM1-0-LL_piControl_r1i1p1f2_gr_196001-305912.nc',
  '1pctCO2': '/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_UKESM1-0-LL_1pctCO2_r1i1p1f2_gr_185001-199912.nc'},
 '3': {'piControl': '/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_NorESM2-LM_piControl_r1i1p1f1_gr_160001-210012.nc',
  '1pctCO2': '/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_NorESM2-LM_1pctCO2_r1i1p1f1_gr_000101-015012.nc'},
 '4': {'piControl': '/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_MIROC-ES2L_piControl_r1i1p1f2_gr_185001-234912.nc',
  '1pctCO2': '/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_MIROC-ES2L_1pctCO2_r1i1p1f2_gr_185001-199912.nc'},
 '5': {'piControl': '/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_CNRM-ESM2-1_piControl_r1i1p1f2_gr_185001-234912.nc',
  '1pctCO2': '/data/ekoehn/CMIP6_derived/talk1/1pctCO2/Omon/talk1_Omon_CNRM-ESM2-1_1pctCO2_r1i1p1f2_gr_

In [6]:
def add_one_month(t):
    return pd.Timestamp(t) + relativedelta(months=1)
    

for key in run_params.keys():

    rparam = run_params[key]

    # get the temporal resolution
    temporal_resolution = get_temporal_resolution(rparam,rparam.experiment,variable)
    if temporal_resolution == 'mon':
        res_factor = 12
        freq = 'MS'
    elif temporal_resolution == 'yr':
        res_factor = 1
        freq = 'YS'

    # get info from 1pctCO2 run about parent  
    with xr.open_dataset(raw_data_paths[key]['1pctCO2']) as ds_1pctco2:
        branch_time_in_parent = ds_1pctco2.attrs['branch_time_in_parent']
        parent_time_units = ds_1pctco2.attrs['parent_time_units']
        parent_root_year = int(parent_time_units.split('-')[0].split(' ')[-1]) # making use of "-" and " " placements in units attribute to isolate reference year
        parent_variant_label = ds_1pctco2.attrs['parent_variant_label']

    # get the piControl calendar to compute the branch timing. First make sure that i get the right member.
    assert parent_variant_label in raw_data_paths[key]['piControl']
    with Dataset(raw_data_paths[key]['piControl']) as ds_pic:
        calendar = ds_pic.variables['time'].calendar
        if calendar == '360_day':
            days_per_year = 360
        elif calendar == '365_day':
            days_per_year = 365
        elif calendar == '366_day':
            days_per_year = 366
        elif calendar == 'gregorian':
            days_per_year = 365.25
        elif calendar == 'proleptic_gregorian':
            days_per_year = 365.25

    # now calculate the numbers of years in the picontrol simulation, after which the 1pctCO2 run was branched
    number_of_years_run_in_piC_from_which_1pctco2_is_branched = int(branch_time_in_parent/days_per_year)
    absolute_year_from_which_1pctco2_is_branched = int(parent_root_year + number_of_years_run_in_piC_from_which_1pctco2_is_branched)

    # print the results
    print(raw_data_paths[key]['piControl'])
    print(f'Take piControl data from {absolute_year_from_which_1pctco2_is_branched:04d} onwards.')

    # now open the piControl file and extract 340 years of data from the branching moment onwards
    numyears_to_extract = 340
    with xr.open_dataset(raw_data_paths[key]['piControl']) as ds_pic:
        
        # get the right numyears_to_extract extracted from the pic file
        year_at_start_of_piC = int(raw_data_paths[key]['piControl'].split('_')[-1].split('-')[0][:4])
        print(year_at_start_of_piC)
        number_of_years_into_run = absolute_year_from_which_1pctco2_is_branched - year_at_start_of_piC
        t0 = number_of_years_into_run * res_factor
        #print(t0)
        
        # now extract the data
        ds = ds_pic.isel(time=slice(t0,t0+numyears_to_extract*res_factor)).squeeze()

        # assert that the correct initial time was chosen 
        assert ds.time.isel(time=0).dt.year == absolute_year_from_which_1pctco2_is_branched
        assert ds.time.isel(time=0).dt.month == 1

        # adjust the time axis to make it uniform across models
        time_pic = pd.date_range(start='1850-01', periods=numyears_to_extract*res_factor, freq=freq)     
        ds['time'] = time_pic

        # Now start the processing. First set up the final dataset
        ds_all = xr.Dataset()
        
        # now calculate annual means 
        annual_means = ds.groupby('time.year').mean('time')#.compute()
        ds_all['annual_means'] = annual_means[variable]
        
        # now calculate the seasonal means and put them into the right shape
        if temporal_resolution == 'mon':
            seasonal_means_dum = ds.resample(time='QS-DEC').mean('time')   # note the December of the first and last year
            seasonal_means_dum['time'] = xr.DataArray([add_one_month(t) for t in seasonal_means_dum['time'].values], dims="time") 
            seasonal_means_dum = seasonal_means_dum.isel(time=slice(None,-1))
            # First: Assign year and season from time
            seasonal_means_dummy = seasonal_means_dum.assign_coords({"year": seasonal_means_dum['time'].dt.year,"season": seasonal_means_dum['time'].dt.season})
            # Then group by year and season separately
            seasonal_means = (seasonal_means_dummy.groupby("year").apply(lambda x: x.groupby("season").first()))  
            # Put the seasonal means into the right order
            seasonal_means = seasonal_means.sel(season=['DJF', 'MAM', 'JJA', 'SON'])
    
            # put them into the final ds_all dataset
            ds_all['seasonal_means']  = seasonal_means[variable]
            
            # now calculate the annual (arg)min and (arg)max and seasonal amplitude (min/max)
            include_min_max = False
            if include_min_max:
                ds_min = ds.groupby('time.year').min(dim='time')
                ds_max = ds.groupby('time.year').max(dim='time')
                ds_all['annual_min']    = ds_min[variable]
                ds_all['annual_max']    = ds_max[variable]
                
                ds_argmin = ds.groupby('time.year').reduce(np.argmin, dim='time')
                ds_argmax = ds.groupby('time.year').reduce(np.argmax, dim='time')
                ds_all['annual_argmin'] = ds_argmin[variable].astype('int8')
                ds_all['annual_argmax'] = ds_argmax[variable].astype('int8')
    
        else:
            print('No seasonality to be computed')
    
        # now add some attributes to the final dataset
        ds_all.attrs['variable'] = variable
        ds_all.attrs['depth_to_analyze'] = depth_to_analyze
        ds_all.attrs['unit'] = ds_pic[variable].attrs['units']
        ds_all.attrs['author'] = 'Eike E. Köhn'
        ds_all.attrs['creation_date'] = date.today().isoformat()
        ds_all.attrs['input_files'] = [raw_data_paths[key]['piControl']]
    
        #print(ds_all)
        if include_min_max == True:
            fsc = 1
        else:
            fsc = 0.66
        print(f'should have a size of about {630*fsc} MB')
    
        # now save the dataset
        save_dir = f'/data/ekoehn/projects/arctic_acidification_reversibility/data/processed_data_piControl/{variable}{depth_to_analyze}'
        os.makedirs(save_dir, exist_ok=True)
        save_filename = f'{variable}{depth_to_analyze}_{rparam.model}_processed.nc'
        ds_all.to_netcdf(f'{save_dir}/{save_filename}')

/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_UKESM1-0-LL_piControl_r1i1p1f2_gr_196001-305912.nc
Take piControl data from 1960 onwards.
1960
should have a size of about 415.8 MB
/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_NorESM2-LM_piControl_r1i1p1f1_gr_160001-210012.nc
Take piControl data from 1600 onwards.
1600
should have a size of about 415.8 MB
/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_MIROC-ES2L_piControl_r1i1p1f2_gr_185001-234912.nc
Take piControl data from 1850 onwards.
1850
should have a size of about 415.8 MB
/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_CNRM-ESM2-1_piControl_r1i1p1f2_gr_185001-234912.nc
Take piControl data from 1850 onwards.
1850
should have a size of about 415.8 MB
/data/ekoehn/CMIP6_derived/talk1/piControl/Omon/talk1_Omon_ACCESS-ESM1-5_piControl_r1i1p1f1_gr_010101-060012.nc
Take piControl data from 0101 onwards.
101
should have a size of about 415.8 MB
/data/ekoehn/CMIP6_derived/talk1/piControl/Omo

In [18]:
absolute_year_from_which_1pctco2_is_branched

501

In [17]:
ds.time.isel(time=0).dt.year

<xarray.DataArray 'year' ()>
array(701)
Coordinates:
    time     object 0701-01-15 12:00:00
    lev      float64 500.0
Attributes:
    standard_name:  time
    bounds:         time_bnds
    axis:           T

## Plot a global mean field (not area weighted) for the 1pctCO2 and the piControl data to show that they start from the same place


In [ ]:
for key in run_params.keys():
    rparam = run_params[key]

    # make the plot for the respective model
    ds_dummy = xr.open_dataset(f'/data/ekoehn/projects/arctic_acidification_reversibility/data/processed_data_1pctCO2-cdr/{variable}{depth_to_analyze}/{variable}{depth_to_analyze}_{rparam.model}_processed.nc')
    ds_pic_dummy = xr.open_dataset(f'/data/ekoehn/projects/arctic_acidification_reversibility/data/processed_data_piControl/{variable}{depth_to_analyze}/{variable}{depth_to_analyze}_{rparam.model}_processed.nc')
    fig,ax = plt.subplots()
    ds_dummy.mean(dim=('lat','lon')).annual_means.plot()
    ds_pic_dummy.mean(dim=('lat','lon')).annual_means.plot()
    #plt.xlim([1850,1900])
    plt.show()
